# NASA C-MAPSS (Turbofan) 엔진 RUL 예측 — 단계별 코드 템플릿
비전공자용: **위에서 아래로 순서대로 실행**하면 전체 파이프라인이 돌아가도록 만들었습니다.

- 데이터: Kaggle `behrad3d/nasa-cmaps`
- 목표: 센서 시계열로 Remaining Useful Life(RUL) 예측
- 평가: RMSE / MAE

> ⚠️ 핵심: 엔진(unit) 단위 시계열이라 무작정 섞어 나누면 누수 위험이 있습니다. 이 노트북은 `unit_number` 기반 분할을 사용합니다.


## 0) 환경 준비
- 라이브러리 불러오기
- 경로 설정

> Kaggle Notebook이면 `DATA_DIR = '/kaggle/input/.../'` 형태로 잡아주세요.
> 로컬/코랩이면 압축 푼 폴더 경로로 바꾸면 됩니다.

In [ ]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

RANDOM_STATE = 42

# ✅ 여기만 본인 환경에 맞게 바꾸세요.
DATA_DIR = "./"  # 예: Kaggle -> "/kaggle/input/nasa-cmaps/"
print("DATA_DIR:", os.path.abspath(DATA_DIR))
print("Files:", os.listdir(DATA_DIR)[:30])


## 1) 데이터 불러오기 (C-MAPSS txt 형식)
원본 C-MAPSS는 `train_FD001.txt`, `test_FD001.txt`, `RUL_FD001.txt` 같이 공백으로 구분된 txt가 많습니다.

아래는 **FD001 기준 템플릿**이고, 다른 FD도 동일하게 적용 가능합니다.

In [ ]:
# 파일명은 데이터셋에 따라 다를 수 있어요. 목록을 보고 정확히 맞춰주세요.
TRAIN_FILE = r"C:\Users\yuzhd\Desktop\유진\data\CMaps\train_FD002.txt"
TEST_FILE  = r"C:\Users\yuzhd\Desktop\유진\data\CMaps\test_FD002.txt"
RUL_FILE   = r"C:\Users\yuzhd\Desktop\유진\data\CMaps\RUL_FD002.txt"

train_path = os.path.join(DATA_DIR, TRAIN_FILE)
test_path  = os.path.join(DATA_DIR, TEST_FILE)
rul_path   = os.path.join(DATA_DIR, RUL_FILE)

col_names = (
    ["unit_number", "time_in_cycles"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

def read_cmapss_txt(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, names=col_names)
    # 어떤 파일은 마지막에 공백 컬럼이 생길 수 있어요(전부 NaN). 그런 컬럼 제거
    df = df.dropna(axis=1, how="all")
    return df

train_df = read_cmapss_txt(train_path)
test_df  = read_cmapss_txt(test_path)
rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])


print("train:", train_df.shape, "test:", test_df.shape)
display(train_df.head())


## 2) Train RUL 만들기
Train은 각 엔진(unit)이 **고장날 때까지** 기록되어 있어서,
`(각 엔진의 마지막 사이클) - (현재 사이클)`로 RUL을 만들 수 있어요.

In [ ]:
max_cycle = train_df.groupby("unit_number")["time_in_cycles"].max().rename("max_cycle")

train_df = train_df.merge(max_cycle, on="unit_number", how="left")
train_df["RUL"] = train_df["max_cycle"] - train_df["time_in_cycles"]
train_df = train_df.drop(columns=["max_cycle"])

display(train_df[["unit_number","time_in_cycles","RUL"]].head(10))
print("RUL min/max:", train_df["RUL"].min(), train_df["RUL"].max())


## 3) Test 정답(RUL) 불러오기 & Test의 현재 RUL 만들기
Test는 각 엔진이 **고장 전 일부 구간만** 제공됩니다.
`RUL_FD001.txt`에는 각 test 엔진의 **마지막 관측 시점에서 남은 수명**이 1개씩 들어있어요.

따라서 test에서 각 row의 RUL은:
- `(engine_last_cycle - time_in_cycles) + RUL_at_last_observed`
로 계산합니다.

In [ ]:
import numpy as np
import pandas as pd

# test 엔진별 "마지막 관측 시점에서 남은 수명" (엔진별 1개씩)
rul_at_last = pd.read_csv(rul_path, header=None, names=["RUL_at_last"])
rul_at_last["unit_number"] = np.arange(1, len(rul_at_last) + 1)

# test 엔진별 마지막 관측 cycle
test_last_cycle = test_df.groupby("unit_number")["time_in_cycles"].max().rename("last_cycle")

test_df = test_df.merge(test_last_cycle, on="unit_number", how="left")
test_df = test_df.merge(rul_at_last, on="unit_number", how="left")

# 각 row의 RUL 계산
test_df["RUL"] = (test_df["last_cycle"] - test_df["time_in_cycles"]) + test_df["RUL_at_last"]

test_df = test_df.drop(columns=["last_cycle", "RUL_at_last"])

print("test_df shape:", test_df.shape)
display(test_df[["unit_number", "time_in_cycles", "RUL"]].head(10))
print("Test RUL min/max:", test_df["RUL"].min(), test_df["RUL"].max())


## 4) 빠른 EDA 체크(필수)
- 결측치
- 센서 상관(샘플)
- 엔진별 길이(사이클 수)

In [ ]:
print("결측치(상위 10개):")
display(train_df.isnull().sum().sort_values(ascending=False).head(10))

print("엔진별 길이(사이클 수) 통계:")
lens = train_df.groupby("unit_number")["time_in_cycles"].max()
display(lens.describe())

sensor_cols = [c for c in train_df.columns if c.startswith("sensor_")]
corr = train_df[sensor_cols].sample(n=min(5000, len(train_df)), random_state=RANDOM_STATE).corr()

plt.figure(figsize=(10,6))
plt.imshow(corr.values, aspect="auto")
plt.title("Sensor correlation (sample)")
plt.colorbar()
plt.show()


## 5) 분할 전략 (누수 방지)
**엔진(unit) 단위로 train/valid 분리**가 기본입니다.

- `GroupShuffleSplit`로 엔진 자체를 분리

In [ ]:
target_col = "RUL"
feature_cols = [c for c in train_df.columns if c != target_col]

X = train_df[feature_cols].copy()
y = train_df[target_col].copy()
groups = train_df["unit_number"].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
groups_train = groups.iloc[train_idx]

print("X_train:", X_train.shape, "X_val:", X_val.shape)
print("train engines:", X_train["unit_number"].nunique(), "val engines:", X_val["unit_number"].nunique())


## 6) 전처리 파이프라인
- 결측치 처리(있다면)
- 스케일링
- Pipeline으로 전처리+모델 묶기

In [ ]:
# unit_number는 ID 성격이라 보통 모델 입력에서 제외 권장
num_cols = [c for c in X_train.columns if c != "unit_number"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocess = ColumnTransformer(
    transformers=[("num", numeric_transformer, num_cols)],
    remainder="drop"
)

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


## 7) 베이스라인 2종
1) DummyRegressor(평균 예측)
2) Ridge(가벼운 선형모델)

In [ ]:
dummy = Pipeline(steps=[("preprocess", preprocess),
                       ("model", DummyRegressor(strategy="mean"))])
dummy.fit(X_train, y_train)
pred = dummy.predict(X_val)
print("Dummy  MAE:", mean_absolute_error(y_val, pred))
print("Dummy RMSE:", rmse(y_val, pred))

ridge = Pipeline(steps=[("preprocess", preprocess),
                       ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE))])
ridge.fit(X_train, y_train)
pred = ridge.predict(X_val)
print("Ridge  MAE:", mean_absolute_error(y_val, pred))
print("Ridge RMSE:", rmse(y_val, pred))


## 8) 랜덤포레스트로 업그레이드(비선형)
- 성능 개선용
- 피처 중요도 확인 가능

In [ ]:
rf = Pipeline(steps=[("preprocess", preprocess),
                    ("model", RandomForestRegressor(
                        n_estimators=300,
                        random_state=RANDOM_STATE,
                        n_jobs=-1
                    ))])
rf.fit(X_train, y_train)
pred = rf.predict(X_val)
print("RF   MAE:", mean_absolute_error(y_val, pred))
print("RF  RMSE:", rmse(y_val, pred))


## 9) 교차검증 (엔진 기준)
GroupKFold로 엔진 단위 CV를 돌려 성능이 '운'인지 확인합니다.

In [ ]:
cv = GroupKFold(n_splits=5)
scores = cross_val_score(
    ridge, X, y,
    cv=cv,
    groups=groups,
    scoring="neg_root_mean_squared_error"
)
print("Ridge CV RMSE:", (-scores).mean(), "+/-", (-scores).std())


## 10) 하이퍼파라미터 튜닝 (예: Ridge alpha)
GridSearchCV도 엔진 기준(GroupKFold)으로 합니다.

In [ ]:
param_grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 50.0]}

gs = GridSearchCV(
    ridge,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
gs.fit(X, y, groups=groups)

print("Best params:", gs.best_params_)
print("Best CV RMSE:", -gs.best_score_)


## 11) 시계열을 더 잘 쓰는 쉬운 방법: '최근 구간 통계 피처' (추천)
시계열(여러 행)을 **엔진 1행(요약 통계)**로 바꾸면 이해가 쉬워지고 누수도 줄어듭니다.

- 마지막 `WINDOW` 사이클의 평균/표준편차/최소/최대 등을 만들기
- 타깃은 마지막 관측 시점의 RUL

In [ ]:
WINDOW = 30
base_cols = [c for c in train_df.columns if c.startswith("op_setting_") or c.startswith("sensor_")]

def make_last_window_stats(df: pd.DataFrame, window: int = 30) -> pd.DataFrame:
    feats = []
    for unit, g in df.groupby("unit_number"):
        g = g.sort_values("time_in_cycles")
        last = g.tail(window)
        agg = {"unit_number": unit}
        for c in base_cols:
            agg[f"{c}_mean"] = last[c].mean()
            agg[f"{c}_std"]  = last[c].std()
            agg[f"{c}_min"]  = last[c].min()
            agg[f"{c}_max"]  = last[c].max()
        agg["RUL"] = g["RUL"].iloc[-1]
        feats.append(agg)
    return pd.DataFrame(feats)

train_tab = make_last_window_stats(train_df, WINDOW)
test_tab  = make_last_window_stats(test_df, WINDOW)

print("train_tab:", train_tab.shape, "test_tab:", test_tab.shape)
display(train_tab.head())


## 12) (엔진 1행 테이블)로 다시 학습

In [ ]:
X2 = train_tab.drop(columns=["RUL"])
y2 = train_tab["RUL"]
groups2 = train_tab["unit_number"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr2, va2 = next(gss.split(X2, y2, groups=groups2))

X2_train, X2_val = X2.iloc[tr2], X2.iloc[va2]
y2_train, y2_val = y2.iloc[tr2], y2.iloc[va2]

num_cols2 = [c for c in X2.columns if c != "unit_number"]
pre2 = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]),
     num_cols2)
], remainder="drop")

model2 = Pipeline([("preprocess", pre2),
                   ("model", RandomForestRegressor(
                       n_estimators=600, random_state=RANDOM_STATE, n_jobs=-1
                   ))])

model2.fit(X2_train, y2_train)
pred2 = model2.predict(X2_val)

print("Engine-table RF  MAE:", mean_absolute_error(y2_val, pred2))
print("Engine-table RF RMSE:", rmse(y2_val, pred2))


## 13) 최종 학습 → Test 예측 → CSV 저장

In [ ]:
final_model = model2.fit(X2, y2)
test_pred = final_model.predict(test_tab.drop(columns=["RUL"]))

submission = pd.DataFrame({
    "unit_number": test_tab["unit_number"],
    "RUL_pred": test_pred
}).sort_values("unit_number")

display(submission.head())
print("pred min/max:", submission["RUL_pred"].min(), submission["RUL_pred"].max())

out_csv = "submission_fd001_template.csv"
submission.to_csv(out_csv, index=False)
print("Saved:", out_csv)


## 14) (선택) 모델 저장/불러오기

In [ ]:
import joblib
joblib.dump(final_model, "final_model.joblib")
loaded = joblib.load("final_model.joblib")
print("Loaded ok. sample pred:", loaded.predict(test_tab.drop(columns=["RUL"]).iloc[:3]))


## 15) 다음에 하면 좋은 확장
1) 상수 센서(분산 0) 제거
2) rolling/lag 피처 추가
3) XGBoost/LightGBM 적용
4) 딥러닝(LSTM) Sliding Window 방식

원하면 너의 실행 결과(파일 목록/에러)를 기준으로 이 템플릿을 네 환경에 맞게 딱 고쳐줄게.